# Web aplikacija i MongoDB baza – Kupina

## 1. Uvod

Kupina je web aplikacija za pretragu i preporučivanje filmova razvijena u okviru projekta iz predmeta Obrada velikog skupa podataka.

Prvi deo projekta obuhvata Flask web aplikaciju i MongoDB bazu podataka. Aplikacija korisnicima omogućava registraciju, prijavu, pregled i pretragu filmova, ocenjivanje filmova, dodavanje filmova u listu za gledanje, ostavljanje komentara i pregled personalizovanih preporuka.

MongoDB se koristi za čuvanje početnog MovieLens skupa podataka, podataka koje generišu korisnici aplikacije, kao i rezultata Spark analiza i ALS sistema preporuke.

## 2. Arhitektura sistema

Tok podataka u projektu je sledeći:

```text
MovieLens skup podataka
        |
        v
MongoDB baza
        |
        v
Apache Spark
  - analiza podataka
  - ALS model
        |
        v
MongoDB kolekcije
  - spark_analytics
  - recommendations
  - als_model_metrics
        |
        v
Flask web aplikacija
        |
        v
Korisnik

In [7]:
from pymongo import MongoClient
from pprint import pprint
import pandas as pd

MONGODB_URI = "mongodb://127.0.0.1:27017"
DATABASE_NAME = "kupina"

client = MongoClient(MONGODB_URI)
db = client[DATABASE_NAME]

print("Povezivanje sa MongoDB bazom je uspešno.")
print("Kolekcije u bazi:")
print(db.list_collection_names())

Povezivanje sa MongoDB bazom je uspešno.
Kolekcije u bazi:
['recommendations', 'ratings', 'als_model_metrics', 'users', 'spark_analytics', 'comments', 'app_users', 'movies', 'watchlist']


## 3. Struktura MongoDB baze

Baza `kupina` sadrži sledeće kolekcije:

- `movies` – podaci o filmovima;
- `ratings` – ocene iz MovieLens skupa i ocene korisnika aplikacije;
- `users` – korisnici iz MovieLens skupa;
- `app_users` – registrovani korisnici web aplikacije;
- `comments` – komentari korisnika;
- `watchlist` – filmovi koje su korisnici sačuvali za kasnije gledanje;
- `recommendations` – personalizovane preporuke generisane ALS modelom;
- `als_model_metrics` – rezultati evaluacije ALS modela;
- `spark_analytics` – rezultati Spark analiza.

In [8]:
collection_counts = []

for collection_name in db.list_collection_names():
    count = db[collection_name].count_documents({})
    collection_counts.append({
        "Kolekcija": collection_name,
        "Broj dokumenata": count
    })

counts_df = pd.DataFrame(collection_counts)
counts_df.sort_values("Broj dokumenata", ascending=False)

,Kolekcija,Broj dokumenata
1,ratings,1000220
0,recommendations,60420
3,users,6040
7,movies,3883
4,spark_analytics,8
8,watchlist,4
6,app_users,3
5,comments,3
2,als_model_metrics,1


Tabela prikazuje broj dokumenata u svakoj kolekciji. Najveće kolekcije su očekivano `ratings`, `movies` i `users`, jer sadrže početni MovieLens skup podataka.

## 4. Pregled podataka o filmovima

In [9]:
movies = list(
    db.movies.find(
        {},
        {
            "_id": 0,
            "movieId": 1,
            "title": 1,
            "genres": 1
        }
    ).limit(10)
)

pd.DataFrame(movies)

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


## 5. Pregled ocena korisnika

In [10]:
ratings = list(
    db.ratings.find(
        {},
        {
            "_id": 0,
            "userId": 1,
            "movieId": 1,
            "rating": 1,
            "timestamp": 1
        }
    ).limit(10)
)

pd.DataFrame(ratings)

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
5,1,1197,3,978302268
6,1,1287,5,978302039
7,1,2804,5,978300719
8,1,594,4,978302268
9,1,919,4,978301368


## 6. Registrovani korisnici aplikacije

Kolekcija `app_users` sadrži korisnike registrovane kroz Flask aplikaciju. Lozinke se ne čuvaju u otvorenom obliku, već kao hash vrednosti.

In [11]:
app_users = list(
    db.app_users.find(
        {},
        {
            "_id": 0,
            "username": 1,
            "email": 1,
            "als_userId": 1
        }
    ).limit(10)
)

pd.DataFrame(app_users)

,username,email,als_userId
0,TestUser,test@gmail.com,6041
1,Natalija,natalija@gmail.com,6042
2,Aleksa,aleksa@gmail.com,6043


## 7. Komentari korisnika

In [18]:
comments = list(
    db.comments.find(
        {},
        {
            "_id": 0,
            "userId": 1,
            "username": 1,
            "movieId": 1,
            "text": 1,
            "createdAt": 1
        }
    ).limit(10)
)

pd.DataFrame(comments)

,userId,username,movieId,text,createdAt
0,6042,Natalija,5,Ovo je dobar film.,2026-07-10 23:05:40.264
1,6042,Natalija,5,Isprobavam funkcionalnost.,2026-07-10 23:06:19.458
2,6043,Aleksa,356,Ovo je moj omiljeni film!,2026-07-19 17:07:04.218


## 8. Lista za gledanje

In [13]:
watchlist = list(
    db.watchlist.find(
        {},
        {
            "_id": 0,
            "userId": 1,
            "movieId": 1,
            "createdAt": 1
        }
    ).limit(10)
)

pd.DataFrame(watchlist)

,userId,movieId
0,6042,3
1,6042,6
2,6042,10
3,6043,10


## 9. Personalizovane preporuke

Preporuke generiše ALS model iz Spark MLlib biblioteke. Rezultati se čuvaju u kolekciji `recommendations`, a Flask aplikacija ih učitava i prikazuje korisniku.

In [14]:
recommendations = list(
    db.recommendations.find(
        {},
        {
            "_id": 0,
            "userId": 1,
            "movieId": 1,
            "position": 1,
            "prediction": 1
        }
    ).sort("position", 1).limit(20)
)

pd.DataFrame(recommendations)

,userId,movieId,position
0,12,572,1
1,6,572,1
2,44,572,1
3,20,572,1
4,16,1851,1
5,27,557,1
6,40,572,1
7,57,2309,1
8,28,557,1
9,31,572,1


## 10. Metrike ALS modela

In [15]:
metrics = list(
    db.als_model_metrics.find(
        {},
        {"_id": 0}
    ).limit(10)
)

pd.DataFrame(metrics)

,model,rank,maxIter,regParam,rmse,trainingRatings,testRatings,evaluatedPredictions,generatedAt
0,ALS,10,10,0.1,0.876048,800454,199766,199727,2026-07-19 15:13:25.771


## 11. Rezultati Spark analiza

Rezultati analiza dobijeni pomoću Apache Spark-a čuvaju se u kolekciji `spark_analytics`. Flask aplikacija ih preuzima i prikazuje u obliku tabela i grafikona.

In [16]:
spark_results = list(
    db.spark_analytics.find(
        {},
        {"_id": 0}
    ).limit(20)
)

pd.DataFrame(spark_results)

,analysis,generatedAt,data
0,most_rated_movies,2026-07-19 15:17:00.897,"[{'movieId': 2858, 'title': 'American Beauty (..."
1,top_rated_movies,2026-07-19 15:17:11.913,"[{'movieId': 2905, 'title': 'Sanjuro (1962)', ..."
2,most_active_users,2026-07-19 15:17:16.501,"[{'userId': 4169, 'number_of_ratings': 2314}, ..."
3,matrix_sparsity,2026-07-19 15:17:16.536,"{'numberOfUsers': 6042, 'numberOfMovies': 3706..."
4,user_activity_distribution,2026-07-19 15:17:21.038,"[{'activity_range': '251-500', 'number_of_user..."
5,movies_by_decade,2026-07-19 15:17:22.110,"[{'decade': 1910, 'number_of_movies': 3}, {'de..."
6,average_rating_by_genre,2026-07-19 15:17:34.207,"[{'genre': 'Film-Noir', 'average_rating': 4.07..."
7,rating_count_vs_average,2026-07-19 15:17:39.006,"[{'movieId': 2858, 'number_of_ratings': 3428, ..."


## 12. Zaključak

Prvi deo projekta obuhvata funkcionalnu Flask web aplikaciju povezanu sa MongoDB bazom. Aplikacija omogućava rad sa filmovima, korisnicima, ocenama, komentarima i listama za gledanje.

Posebna vrednost projekta je integracija sa Spark delom, jer se rezultati analiza i preporuke generisane ALS modelom čuvaju u MongoDB i prikazuju direktno kroz web aplikaciju.